# 03 - Bronze to Silver to Gold

Deduplicates at-least-once stream delivery by event_id before merging Silver.

In [ ]:
import os,re
from pyspark.sql import Window, functions as F
from pyspark.sql.types import *
catalog=os.getenv('AIDP_CATALOG','aidp_poc')
if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*',catalog): raise ValueError('AIDP_CATALOG must be a simple Spark identifier')
schema=StructType([StructField('schema_version',StringType()),StructField('event_id',StringType()),StructField('event_type',StringType()),StructField('event_ts_utc',TimestampType()),StructField('meter_id',StringType()),StructField('device_id',StringType()),StructField('service_point_id',StringType()),StructField('service_point_type',StringType()),StructField('interval_start_utc',TimestampType()),StructField('interval_end_utc',TimestampType()),StructField('interval_minutes',IntegerType()),StructField('consumption_kwh',DoubleType()),StructField('demand_kw',DoubleType()),StructField('voltage_v',DoubleType()),StructField('current_a',DoubleType()),StructField('power_factor',DoubleType()),StructField('frequency_hz',DoubleType()),StructField('temperature_c',DoubleType()),StructField('humidity_pct',DoubleType()),StructField('quality_code',StringType()),StructField('tariff_code',StringType()),StructField('meter_status',StringType()),StructField('firmware_version',StringType()),StructField('region_code',StringType()),StructField('feeder_id',StringType()),StructField('transformer_id',StringType()),StructField('customer_segment',StringType()),StructField('outage_minutes',IntegerType()),StructField('tamper_flag',BooleanType()),StructField('measurement_events',ArrayType(StringType())),StructField('device_events',ArrayType(StringType()))])
bronze,quarantine,silver=f'{catalog}.bronze.meter_reading',f'{catalog}.bronze.meter_reading_quarantine',f'{catalog}.silver.interval_reading'
gold_interval,gold_daily=f'{catalog}.gold.meter_interval_usage',f'{catalog}.gold.service_point_daily_usage'


In [ ]:
raw=spark.table(bronze).select('stream_partition','stream_offset','stream_key','raw_value')
checked=raw.withColumn('payload',F.from_json('raw_value',schema)).withColumn('ingested_at',F.current_timestamp()); p=F.col('payload')
checked=checked.withColumn('validation_reason',F.concat_ws('; ',F.when(p.isNull(),'INVALID_JSON_OR_SCHEMA'),F.when(p.schema_version!='2.0','UNSUPPORTED_SCHEMA_VERSION'),F.when(p.event_id.isNull()|p.meter_id.isNull()|p.interval_start_utc.isNull()|p.consumption_kwh.isNull()|p.demand_kw.isNull(),'MISSING_REQUIRED_FIELD'),F.when(p.event_type!='INTERVAL_READING','UNSUPPORTED_EVENT_TYPE'),F.when(p.interval_minutes!=15,'INVALID_INTERVAL_MINUTES'),F.when((p.consumption_kwh<0)|(p.consumption_kwh>1000)|(p.demand_kw<0),'INVALID_CONSUMPTION_OR_DEMAND'),F.when((p.power_factor < -1)|(p.power_factor > 1),'INVALID_POWER_FACTOR'),F.when(p.interval_end_utc != p.interval_start_utc + F.expr('INTERVAL 15 MINUTES'),'INVALID_INTERVAL_BOUNDARY'),F.when(~p.quality_code.isin('ACTUAL','ESTIMATED','SUBSTITUTED','SUSPECT'),'INVALID_QUALITY_CODE')))
bad=checked.where("validation_reason <> ''").select('stream_partition','stream_offset','stream_key','raw_value','validation_reason',F.current_timestamp().alias('quarantined_at')); bad.createOrReplaceTempView('bad_batch'); spark.sql(f'MERGE INTO {quarantine} t USING bad_batch s ON t.stream_partition=s.stream_partition AND t.stream_offset=s.stream_offset WHEN NOT MATCHED THEN INSERT *')
good=checked.where("validation_reason = ''").select('payload.*','stream_partition','stream_offset','stream_key','ingested_at',F.to_date('payload.interval_start_utc').alias('reading_date'),(p.quality_code=='ACTUAL').alias('is_actual'))
# OCI delivery is at-least-once: retain one deterministic copy for every business event.
good=good.withColumn('_dedupe_rank',F.row_number().over(Window.partitionBy('event_id').orderBy(F.col('ingested_at').desc(),F.col('stream_offset').desc()))).where('_dedupe_rank=1').drop('_dedupe_rank')
good.createOrReplaceTempView('silver_batch'); spark.sql(f'MERGE INTO {silver} t USING silver_batch s ON t.event_id=s.event_id WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *')


In [ ]:
spark.sql(f'''MERGE INTO {catalog}.silver.device t USING (SELECT device_id,meter_id,service_point_id,service_point_type,max_by(firmware_version,interval_start_utc) firmware_version,max_by(meter_status,interval_start_utc) meter_status,min(interval_start_utc) first_seen_at,max(interval_start_utc) last_seen_at,max_by(event_id,interval_start_utc) last_event_id,current_timestamp() updated_at FROM {silver} GROUP BY device_id,meter_id,service_point_id,service_point_type) s ON t.device_id=s.device_id WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *''')
spark.sql(f'''MERGE INTO {catalog}.silver.service_point t USING (SELECT service_point_id,service_point_type,max_by(meter_id,interval_start_utc) latest_meter_id,max_by(region_code,interval_start_utc) region_code,max_by(feeder_id,interval_start_utc) feeder_id,max_by(transformer_id,interval_start_utc) transformer_id,max_by(customer_segment,interval_start_utc) customer_segment,min(interval_start_utc) first_seen_at,max(interval_start_utc) last_seen_at,current_timestamp() updated_at FROM {silver} GROUP BY service_point_id,service_point_type) s ON t.service_point_id=s.service_point_id WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *''')
spark.sql(f'''MERGE INTO {catalog}.silver.device_event t USING (SELECT event_id,meter_id,device_id,service_point_id,interval_start_utc,explode(device_events) device_event_type,ingested_at,reading_date FROM {silver}) s ON t.event_id=s.event_id AND t.device_event_type=s.device_event_type WHEN NOT MATCHED THEN INSERT *''')
spark.sql(f'''CREATE OR REPLACE TEMP VIEW gold_interval AS SELECT reading_date,interval_start_utc,meter_id,service_point_id,region_code,feeder_id,transformer_id,customer_segment,tariff_code,sum(consumption_kwh) consumption_kwh,max(demand_kw) demand_kw,avg(voltage_v) avg_voltage_v,avg(power_factor) avg_power_factor,count(*) reading_count,sum(CASE WHEN is_actual THEN 1 ELSE 0 END) actual_reading_count,sum(CASE WHEN quality_code='SUSPECT' THEN 1 ELSE 0 END) suspect_reading_count,sum(CASE WHEN tamper_flag THEN 1 ELSE 0 END) tamper_reading_count,sum(CASE WHEN outage_minutes>0 THEN 1 ELSE 0 END) outage_reading_count,current_timestamp() refreshed_at FROM {silver} GROUP BY reading_date,interval_start_utc,meter_id,service_point_id,region_code,feeder_id,transformer_id,customer_segment,tariff_code''')
spark.sql(f'INSERT OVERWRITE {gold_interval} SELECT * FROM gold_interval')
spark.sql(f'''INSERT OVERWRITE {gold_daily} SELECT reading_date,service_point_id,region_code,feeder_id,transformer_id,sum(consumption_kwh),sum(CASE WHEN tariff_code='PEAK' THEN consumption_kwh ELSE 0 END),sum(CASE WHEN tariff_code='OFF_PEAK' THEN consumption_kwh ELSE 0 END),max(demand_kw),count(*),sum(actual_reading_count),sum(tamper_reading_count),sum(CAST(outage_reading_count AS BIGINT))*15,current_timestamp() FROM {gold_interval} GROUP BY reading_date,service_point_id,region_code,feeder_id,transformer_id''')
print('Silver, dimensions, and Gold tables refreshed.')